# Week 3, day 2 — Worksheet 07 SOLUTIONS: transforming values   (L05)

Executed in the lab image (pandas 3.0.5) against the real files in `data/`.
Every quoted number is what it actually printed.

Question 4 is the one to re-read. The missing markers in this file are three
different strings, and finding two of them is worse than finding none.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Transforming values. Run this once.
import numpy as np
import pandas as pd

cust = pd.read_csv("data/customers_messy.csv")
orders = pd.read_csv("data/orders_long.csv")

print("customers:", cust.shape)
print("distinct Province values:", cust["Province"].nunique(dropna=False))
print("Province NaN rows:", int(cust["Province"].isna().sum()))

PART A — replace: exact values you already know

### Question 1

**16** distinct values, including `nan`, `'-'`, `'?'` — and **`'Saskachewan'`**. -> `sorted()` raises `TypeError: '<' not supported between instances of 'str' and 'float'`.

Three things in one printout.

`sorted()` fails because the column mixes strings with a float (`nan`), and
Python refuses to order those. That is not a Pandas quirk — it is why
`repr()` and `isinstance` checks matter on a column you have not cleaned.

`'-'` and `'?'` are there. **`'N/A'` is not**, even though the file was
written with it. Q2 explains where it went.

And `Saskachewan` is missing a `t`. That is a spelling mistake in the source
system, exactly like `Prarie` in the region column, and it is now a fact
about your data rather than something you can filter for correctly.

In [ ]:
vals = cust["Province"].unique()
print(len(vals), "distinct values:")
for v in vals:
    print("  ", repr(v))

print()
# sorted() raises: the column mixes strings with a float (nan), and Python
# will not order those against each other.
try:
    sorted(vals)
except Exception as exc:
    print("sorted(vals) -> %s: %s" % (type(exc).__name__, exc))
print("sorted, skipping NaN:", sorted(v for v in vals if isinstance(v, str))[:4], "...")

### Question 2

As text: `-` **17**, `?` **16**. `NaN`: **17**. -> with `keep_default_na=False`: `N/A` **17**, `-` **17**, `?` **16**.

`N/A` was converted to `NaN` while the file was being read.

`read_csv` has a built-in list of strings it treats as missing — `N/A`,
`NA`, `null`, `NaN`, `nan`, `#N/A`, `-nan`, an empty field, and about a
dozen more. `N/A` is on it; `-` and `?` are not.

So of three markers meaning exactly the same thing, one was handled for you
silently and two survived as ordinary text. That is worse than handling
none, because the column now contains **two different encodings of
missing** and any check you write will find one of them.

`keep_default_na=False` turns the magic off and shows you the file as
written. Worth doing once on any new source, just to see what was being
decided for you.

In [ ]:
markers = ["N/A", "-", "?"]
counts = cust["Province"].value_counts()
print("markers still present as text:")
print(counts[counts.index.isin(markers)].to_string())
print()
print("NaN rows:", int(cust["Province"].isna().sum()))
print()
# Read it again with the default NA handling switched off:
raw = pd.read_csv("data/customers_messy.csv", keep_default_na=False)
print("with keep_default_na=False:")
print(raw["Province"].value_counts()[["N/A", "-", "?"]].to_string())

### Question 3

`NaN` goes from **17** to **50** after replacing all three markers. -> 13 real province values remain.

33 more rows recognised as missing — the 17 dashes and 16 question marks.

50 of 440 is 11% of the column, and before this line 33 of those looked
like legitimate values to every count, filter and `groupby` you might run.

In [ ]:
before = cust["Province"].isna().sum()
fixed = cust["Province"].replace(["N/A", "-", "?"], np.nan)
print("NaN before:", before)
print("NaN after: ", fixed.isna().sum())
print()
print("distinct values now:", len(fixed.dropna().unique()))

### Question 4

Partial replace -> **33** `NaN`, and `'-'` still present. -> `dropna()` would remove 33 rather than 50, silently keeping **17** rows with a bad value.

This is the question. A partial clean-up is more dangerous than none.

After the full replace, the column is honest: 50 rows are missing and
`dropna()`, `isna()` and every aggregate will treat them consistently.

After the partial one, the column *looks* cleaner — the `?`s are gone, the
count went up, the work appears done — and 17 rows still carry `'-'` as if
it were a province. `value_counts()` will list it. A `groupby("Province")`
will give it its own group. A join will not match it to anything.

The defence is to enumerate the distinct values (Q1) rather than fixing the
markers you happen to remember.

In [ ]:
partial = cust["Province"].replace(["N/A", "?"], np.nan)
print("NaN after partial replace:", partial.isna().sum())
print("still-present markers:",
      [v for v in partial.dropna().unique() if v in ("N/A", "-", "?")])
print()
full = cust["Province"].replace(["N/A", "-", "?"], np.nan)
print("rows dropna() would remove, partial:", partial.isna().sum())
print("rows dropna() would remove, full:   ", full.isna().sum())
print("rows silently kept with a bad value:",
      full.isna().sum() - partial.isna().sum())

### Question 5

Before: `Corporate 173`, `Consumer 101`, `Home Office 89`, `Small Business 77`. -> after `replace`, the two named values change and the other two are untouched.

`replace` is a targeted swap. Values you did not mention keep whatever they
had, which is almost always what you want when standardising a few labels.

In [ ]:
print("before:")
print(cust["Segment"].value_counts().to_string())
print()
after = cust["Segment"].replace({"Small Business": "SMB", "Home Office": "HO"})
print("after replace:")
print(after.value_counts().to_string())

### Question 6

`map` with the same dictionary -> **`NaN` 274**, `HO 89`, `SMB 77`. -> `map` produced **274** `NaN`; `replace` produced **0**.

Same column, same dictionary, and `map` destroyed 62% of it.

The two have opposite defaults for the values you did not list.
**`replace` keeps them; `map` discards them.** `Corporate` and `Consumer`
were not in the dictionary, so all 274 of those rows became missing.

Neither is wrong. `map` is a total translation — every value must have an
entry, and a gap is a real problem you want surfaced. `replace` is a partial
edit — you are fixing some values and leaving the rest.

Choose by which of those you mean. And when you use `map`, run the
set-difference check from the week 3, day 1 class first.

In [ ]:
mapped = cust["Segment"].map({"Small Business": "SMB", "Home Office": "HO"})
print(mapped.value_counts(dropna=False).to_string())
print()
print("NaN produced by map:", mapped.isna().sum())
print("NaN produced by replace:",
      cust["Segment"].replace({"Small Business": "SMB", "Home Office": "HO"}).isna().sum())

PART B — .loc for conditions, not values

### Question 7

**1** row above `0.10`; max goes from `0.21` to `0.1`.

`.loc[mask, column] = value` writes into the rows the mask selects. This is
the supported way to make a conditional change — chained forms like
`work[mask]["Discount"] = 0.10` write into a temporary and are discarded.

`replace` could not express this. It swaps values you can name, and 'every
value above 0.10' is a range, not a value. **Exact match -> `replace`;
condition or range -> `.loc[]`.**

One row in 1,093, incidentally. Worth knowing before you build a capping
rule that only ever fires once.

In [ ]:
work = orders.copy()
mask = work["Discount"] > 0.10
print("rows above 0.10:", mask.sum())
print("max before:", work["Discount"].max())

work.loc[mask, "Discount"] = 0.10
print("max after: ", work["Discount"].max())
print("rows still above 0.10:", (work["Discount"] > 0.10).sum())

### Question 8

`normal 1092`, `capped 1`. -> the original `Discount` is untouched, max still `0.21`.

The same rule, recorded instead of applied.

Overwriting destroys the evidence: after Q7 there is no way to tell which
row was capped or what it used to be. Marking keeps the original value *and*
records the decision, so someone can audit it, reverse it, or count how
often the rule fires.

On a pipeline that runs nightly, the flag column is also your alarm. If
`capped` jumps from 1 to 400 one morning, something upstream changed.

In [ ]:
work = orders.copy()
work["Flag"] = "normal"
work.loc[work["Discount"] > 0.10, "Flag"] = "capped"
print(work["Flag"].value_counts().to_string())
print()
print("original Discount untouched, max still:", work["Discount"].max())

# Overwriting destroys the evidence. Marking keeps the original value AND
# records the decision, so someone can audit or reverse it later.

### Question 9

`profit 625`, `loss 468`, totalling `1093`. -> **no `break-even` rows at all.**

The three counts add to the row count, which is the check that no case was
missed.

And the `break-even` bucket is empty: not one of the 1,093 orders has a
`Profit` of exactly zero. That matches the week 3, day 1 class, where the same file
had 30 discounts of exactly zero and no profits of exactly zero.

The order of the three `.loc` assignments matters. Starting everything at
`"profit"` and then overwriting the exceptions is only correct because the
conditions are mutually exclusive and applied narrowest-last. Overlapping
conditions in the wrong order give a silently wrong column.

In [ ]:
work = orders.copy()
work["Result"] = "profit"
work.loc[work["Profit"] == 0, "Result"] = "break-even"
work.loc[work["Profit"] < 0, "Result"] = "loss"
counts = work["Result"].value_counts()
print(counts.to_string())
print()
print("total:", counts.sum(), "of", len(work))

### Question 10

`map(lambda p: p.split()[1])` -> **raises** `AttributeError: 'float' object has no attribute 'split'`.

The 17 `NaN`s from Q2 are floats, and floats have no `.split()`.

Put this beside Q6 and the pair is the lesson. `map` with a **dictionary**
met values it did not know and returned `NaN` for 274 rows without a
murmur. `map` with a **function** met one it could not handle and stopped
everything.

Same method, same column, opposite failure modes — decided by whether you
handed it a lookup or a callable. When you pass a function to `map` or
`apply`, it will be called on the missing values too, and it has to survive
that: guard with `if pd.isna(p)`, or use `.dropna()` first, or pass
`na_action="ignore"` to leave the `NaN`s alone.

In [ ]:
one_word = sorted({p for p in cust["Province"].unique()
                   if isinstance(p, str) and len(p.split()) == 1})
print("one-word provinces:", one_word[:5])
print("NaN values in the column:", int(cust["Province"].isna().sum()))
print(cust["Province"].map(lambda p: p.split()[1]))